# Supermarket Sales Analysis — Decision Dashboard

This notebook turns supermarket transactions into a decision-oriented dashboard. It deliberately avoids a raw data dump and progresses from executive KPIs to trends, drivers, risks, and actions.

**Data source:** [Supermarket Sales dataset](https://raw.githubusercontent.com/aungpyaeap/supermarket-sales/master/supermarket_sales%20-%20Sheet1.csv). If the URL is unavailable, the notebook creates a reproducible synthetic dataset with the same analytical columns.

## Setup and analysis configuration

In [1]:
from pathlib import Path
from urllib.error import URLError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
DATA_URL = "https://raw.githubusercontent.com/aungpyaeap/supermarket-sales/master/supermarket_sales%20-%20Sheet1.csv"
OUTPUT_DIRECTORY = Path(".")
KPI_PLOT_PATH = OUTPUT_DIRECTORY / "kpi_summary.png"
PRODUCT_REVENUE_PLOT_PATH = OUTPUT_DIRECTORY / "revenue_by_product.png"

sns.set_theme(style="whitegrid", context="notebook")
CUSTOM_PALETTE = {
    "navy": "#16324F",
    "teal": "#2A9D8F",
    "gold": "#E9C46A",
    "orange": "#F4A261",
    "coral": "#E76F51",
    "slate": "#5C677D",
}
PRODUCT_PALETTE = "crest"
BRANCH_PALETTE = "Set2"

ModuleNotFoundError: No module named 'numpy'

## Data loading with an offline fallback

The loader first attempts the published CSV. The fallback preserves the source schema and derives `Total` from `Unit price × Quantity × (1 + tax)` so that all downstream analysis remains executable.

In [ ]:
def generate_synthetic_sales_data(row_count=1000, random_seed=RANDOM_SEED):
    """Create a deterministic dataset with the standard supermarket schema."""
    rng = np.random.default_rng(random_seed)
    product_lines = np.array([
        "Health and beauty", "Electronic accessories", "Home and lifestyle",
        "Sports and travel", "Food and beverages", "Fashion accessories"
    ])
    branches = np.array(["A", "B", "C"])
    cities = {"A": "Yangon", "B": "Mandalay", "C": "Naypyitaw"}
    dates = pd.date_range("2019-01-01", periods=89, freq="D")
    selected_branches = rng.choice(branches, size=row_count, p=[0.34, 0.31, 0.35])
    selected_products = rng.choice(product_lines, size=row_count)
    quantities = rng.integers(1, 11, size=row_count)
    unit_prices = np.round(rng.uniform(10, 100, size=row_count), 2)
    tax_rates = rng.uniform(0.01, 0.10, size=row_count)
    totals = np.round(unit_prices * quantities * (1 + tax_rates), 2)
    transaction_dates = rng.choice(dates, size=row_count)
    hours = rng.integers(10, 21, size=row_count)
    minutes = rng.integers(0, 60, size=row_count)
    return pd.DataFrame({
        "Invoice ID": [f"SYN-{index:04d}" for index in range(1, row_count + 1)],
        "Branch": selected_branches,
        "City": [cities[branch] for branch in selected_branches],
        "Customer type": rng.choice(["Member", "Normal"], size=row_count),
        "Gender": rng.choice(["Male", "Female"], size=row_count),
        "Product line": selected_products,
        "Unit price": unit_prices,
        "Quantity": quantities,
        "Tax 5%": np.round(totals - unit_prices * quantities, 2),
        "Total": totals,
        "Date": pd.Series(transaction_dates).dt.strftime("%m/%d/%Y"),
        "Time": [f"{hour:02d}:{minute:02d}" for hour, minute in zip(hours, minutes)],
        "Payment": rng.choice(["Cash", "Credit card", "Ewallet"], size=row_count),
        "cogs": np.round(totals * 0.55, 2),
        "gross margin percentage": 0.48,
        "gross income": np.round(totals * 0.48, 2),
        "Rating": np.round(rng.uniform(5.0, 10.0, size=row_count), 1),
    })


def load_sales_data(data_url):
    """Load the source dataset, falling back to synthetic data when unavailable."""
    try:
        loaded_data = pd.read_csv(data_url)
        data_source = "published Supermarket Sales CSV"
    except (URLError, TimeoutError, ValueError, OSError) as error:
        print(f"Online data load failed ({error.__class__.__name__}); using synthetic fallback data.")
        loaded_data = generate_synthetic_sales_data()
        data_source = "synthetic fallback data"
    return loaded_data, data_source


sales_data, data_source = load_sales_data(DATA_URL)
print(f"Loaded {len(sales_data):,} transaction rows from {data_source}.")
display(sales_data.head(3))

## Data quality checks and preprocessing

In [ ]:
required_columns = {
    "Invoice ID", "Branch", "Product line", "Unit price", "Quantity",
    "Total", "Date", "Time", "Rating"
}
missing_required_columns = required_columns.difference(sales_data.columns)
if missing_required_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_required_columns)}")

null_counts = sales_data.isna().sum().sort_values(ascending=False)
duplicate_count = int(sales_data.duplicated().sum())
print("Null values by column:")
display(null_counts[null_counts.gt(0)].to_frame("null_count") if null_counts.gt(0).any() else pd.DataFrame({"status": ["No null values found"]}))
print(f"Duplicate rows found: {duplicate_count:,}")

sales_data = sales_data.drop_duplicates().copy()
sales_data["Date"] = pd.to_datetime(sales_data["Date"], errors="coerce")
sales_data["Time"] = sales_data["Time"].astype(str)
sales_data["Hour"] = pd.to_datetime(sales_data["Time"], format="%H:%M", errors="coerce").dt.hour
sales_data["Month"] = sales_data["Date"].dt.to_period("M").astype(str)
sales_data["Total"] = pd.to_numeric(sales_data["Total"], errors="coerce")
sales_data["Rating"] = pd.to_numeric(sales_data["Rating"], errors="coerce")
sales_data = sales_data.dropna(subset=["Date", "Hour", "Month", "Total", "Rating"])

print(f"Analysis-ready rows: {len(sales_data):,}")
print(f"Date range: {sales_data['Date'].min():%Y-%m-%d} to {sales_data['Date'].max():%Y-%m-%d}")

## LEVEL 1 — Executive KPIs

These metrics answer the first management questions: how much revenue was generated, how many orders occurred, what was the typical order value, and how did customers rate the experience?

In [ ]:
total_revenue = sales_data["Total"].sum()
total_orders = sales_data["Invoice ID"].nunique()
average_order_value = total_revenue / total_orders if total_orders else 0.0
average_rating = sales_data["Rating"].mean()

executive_kpis = pd.Series({
    "Total Revenue": total_revenue,
    "Total Orders": total_orders,
    "Average Order Value": average_order_value,
    "Average Rating": average_rating,
})
display(pd.DataFrame({
    "Metric": executive_kpis.index,
    "Value": [
        f"${total_revenue:,.2f}", f"{total_orders:,}",
        f"${average_order_value:,.2f}", f"{average_rating:.2f} / 10",
    ],
}))

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
kpi_labels = ["Revenue", "Orders", "AOV", "Rating"]
kpi_values = [total_revenue, total_orders, average_order_value, average_rating]
kpi_display_values = [f"${total_revenue:,.0f}", f"{total_orders:,}", f"${average_order_value:,.2f}", f"{average_rating:.2f}/10"]
kpi_colors = [CUSTOM_PALETTE["navy"], CUSTOM_PALETTE["teal"], CUSTOM_PALETTE["gold"], CUSTOM_PALETTE["coral"]]
for axis, label, value, display_value, color in zip(axes.flat, kpi_labels, kpi_values, kpi_display_values, kpi_colors):
    axis.bar([label], [value], color=color, width=0.55)
    axis.set_title(label, fontweight="bold")
    axis.text(0, value, display_value, ha="center", va="bottom", fontweight="bold", fontsize=12)
    axis.set_ylabel("Value")
    axis.tick_params(axis="x", length=0)
    axis.spines[["top", "right"]].set_visible(False)
fig.suptitle("Executive KPI Summary", fontsize=16, fontweight="bold")
plt.tight_layout()
fig.savefig(KPI_PLOT_PATH, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved KPI chart to {KPI_PLOT_PATH.resolve()}")

## LEVEL 2 — Trends

Trend views reveal when demand and revenue concentrate. The monthly view supports planning, while the hourly view supports staffing and opening-hours decisions.

In [ ]:
monthly_revenue = sales_data.groupby("Month", as_index=False)["Total"].sum()
hourly_revenue = sales_data.groupby("Hour", as_index=False)["Total"].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=monthly_revenue, x="Month", y="Total", marker="o", color=CUSTOM_PALETTE["navy"], ax=axes[0])
axes[0].set_title("Monthly Revenue Trend", fontweight="bold")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Revenue ($)")
axes[0].tick_params(axis="x", rotation=45)
sns.barplot(data=hourly_revenue, x="Hour", y="Total", color=CUSTOM_PALETTE["teal"], ax=axes[1])
axes[1].set_title("Revenue by Hour", fontweight="bold")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Revenue ($)")
plt.tight_layout()
plt.show()

peak_month = monthly_revenue.loc[monthly_revenue["Total"].idxmax()]
peak_hour = hourly_revenue.loc[hourly_revenue["Total"].idxmax()]
print(f"Peak month: {peak_month['Month']} (${peak_month['Total']:,.2f})")
print(f"Peak trading hour: {int(peak_hour['Hour']):02d}:00 (${peak_hour['Total']:,.2f})")

## LEVEL 3 — Drivers

Product line and branch performance explain where revenue is being created. The ranked views make concentration and underperformance visible without displaying every transaction.

In [ ]:
product_revenue = sales_data.groupby("Product line", as_index=False)["Total"].sum().sort_values("Total", ascending=False)
branch_revenue = sales_data.groupby("Branch", as_index=False)["Total"].sum().sort_values("Total", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=product_revenue, y="Product line", x="Total", hue="Product line", palette=PRODUCT_PALETTE, legend=False, ax=axes[0])
axes[0].set_title("Revenue by Product Line", fontweight="bold")
axes[0].set_xlabel("Revenue ($)")
axes[0].set_ylabel("")
sns.barplot(data=branch_revenue, x="Branch", y="Total", hue="Branch", palette=BRANCH_PALETTE, legend=False, ax=axes[1])
axes[1].set_title("Revenue by Branch", fontweight="bold")
axes[1].set_xlabel("Branch")
axes[1].set_ylabel("Revenue ($)")
for axis in axes:
    axis.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(PRODUCT_REVENUE_PLOT_PATH, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved driver chart to {PRODUCT_REVENUE_PLOT_PATH.resolve()}")
display(product_revenue.rename(columns={"Total": "Revenue"}).style.format({"Revenue": "${:,.2f}"}))
display(branch_revenue.rename(columns={"Total": "Revenue"}).style.format({"Revenue": "${:,.2f}"}))

## LEVEL 4 — Risk Analysis

Risks are flagged against relative benchmarks so they remain useful across the real and synthetic datasets. A low rating can signal service or product-quality churn risk; a lagging branch can signal a local execution or demand risk.

In [ ]:
overall_rating = sales_data["Rating"].mean()
overall_branch_revenue = sales_data.groupby("Branch")["Total"].sum()
branch_revenue_benchmark = overall_branch_revenue.mean()
product_rating_summary = sales_data.groupby("Product line").agg(
    average_rating=("Rating", "mean"),
    revenue=("Total", "sum"),
    orders=("Invoice ID", "nunique"),
).reset_index()
branch_risk_summary = sales_data.groupby("Branch").agg(
    revenue=("Total", "sum"),
    average_rating=("Rating", "mean"),
    orders=("Invoice ID", "nunique"),
).reset_index()

low_rating_products = product_rating_summary[product_rating_summary["average_rating"] < overall_rating].sort_values("average_rating")
lagging_branches = branch_risk_summary[branch_risk_summary["revenue"] < branch_revenue_benchmark].sort_values("revenue")

risk_findings = []
if not low_rating_products.empty:
    lowest_rating_product = low_rating_products.iloc[0]
    risk_findings.append({
        "Risk": "Customer experience / churn",
        "Evidence": f"{lowest_rating_product['Product line']} averages {lowest_rating_product['average_rating']:.2f}/10 versus {overall_rating:.2f}/10 overall.",
        "Priority": "High" if lowest_rating_product["average_rating"] < overall_rating - 0.25 else "Medium",
    })
else:
    risk_findings.append({"Risk": "Customer experience / churn", "Evidence": "No product line is below the overall rating benchmark.", "Priority": "Monitor"})
if not lagging_branches.empty:
    lowest_revenue_branch = lagging_branches.iloc[0]
    risk_findings.append({
        "Risk": "Branch performance / capacity",
        "Evidence": f"Branch {lowest_revenue_branch['Branch']} generates ${lowest_revenue_branch['revenue']:,.2f}, below the ${branch_revenue_benchmark:,.2f} branch benchmark.",
        "Priority": "High" if lowest_revenue_branch["revenue"] < branch_revenue_benchmark * 0.9 else "Medium",
    })
else:
    risk_findings.append({"Risk": "Branch performance / capacity", "Evidence": "No branch is below the average branch revenue benchmark.", "Priority": "Monitor"})

risk_table = pd.DataFrame(risk_findings)
display(risk_table)
print("Risk interpretation: prioritize findings marked High, then validate root causes with store managers and customer feedback.")

## LEVEL 5 — Actionable Recommendations

The statements below convert measured facts into decisions using the **Fact → Insight → Opportunity → Action** format.

In [ ]:
top_product = product_revenue.iloc[0]
bottom_product = product_revenue.iloc[-1]
top_branch = branch_revenue.iloc[0]

recommendations = [
    f"Fact: {top_product['Product line']} leads product-line revenue at ${top_product['Total']:,.2f}. -> Insight: demand is concentrated in a clear category leader. -> Opportunity: increase basket size around the winning category. -> Action: test two cross-sell bundles and track AOV and bundle conversion for four weeks.",
    f"Fact: {bottom_product['Product line']} is the lowest-revenue product line at ${bottom_product['Total']:,.2f}. -> Insight: assortment, visibility, or pricing may be limiting demand. -> Opportunity: recover incremental revenue without broad discounting. -> Action: run a targeted end-cap and price test, comparing weekly revenue and margin with a control period.",
    f"Fact: Branch {top_branch['Branch']} is the revenue leader at ${top_branch['Total']:,.2f}. -> Insight: its operating practices may be transferable. -> Opportunity: replicate high-performing staffing and merchandising routines. -> Action: document the branch playbook and pilot one practice in the lowest-revenue branch.",
    f"Fact: the highest-revenue hour is {int(peak_hour['Hour']):02d}:00. -> Insight: demand is time-concentrated. -> Opportunity: protect service quality during the peak window. -> Action: schedule a peak-hour staffing experiment and monitor queue time, ratings, and revenue per labor hour.",
]
for number, recommendation in enumerate(recommendations, start=1):
    print(f"{number}. {recommendation}\n")

## Final summary metrics

This compact output is suitable for handoff to an executive dashboard or report.

In [ ]:
summary_metrics = {
    "Data source": data_source,
    "Analysis rows": f"{len(sales_data):,}",
    "Total Revenue": f"${total_revenue:,.2f}",
    "Total Orders": f"{total_orders:,}",
    "Average Order Value": f"${average_order_value:,.2f}",
    "Average Rating": f"{average_rating:.2f} / 10",
    "Peak Month": f"{peak_month['Month']} (${peak_month['Total']:,.2f})",
    "Peak Hour": f"{int(peak_hour['Hour']):02d}:00 (${peak_hour['Total']:,.2f})",
    "Saved plots": f"{KPI_PLOT_PATH.name}, {PRODUCT_REVENUE_PLOT_PATH.name}",
}
for metric_name, metric_value in summary_metrics.items():
    print(f"{metric_name}: {metric_value}")